<a href="https://colab.research.google.com/github/Niseta701/Tech400/blob/main/week6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import argparse
import math
import re
import json
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np

# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_DIR = "/content"       # Folder containing CISI.ALL, CISI.QRY, CISI.REL
K = 10                      # Number of top documents for evaluation
CUSTOM_QUERY = None         # Example: "information retrieval system"
OUTPUT_FILE = "/content/results.json"


# ============================================================
# 2. DATA LOADING
# ============================================================

def parse_cisi_file(path):
    """Parse CISI document/query files into {id: text}."""
    items = {}
    cur_id = None
    field = None
    buf = defaultdict(list)

    with open(path, encoding="latin-1") as f:
        for raw in f:
            line = raw.rstrip("\r\n")

            if line.startswith(".I"):
                if cur_id is not None:
                    items[cur_id] = " ".join(buf["T"] + buf["W"])

                cur_id = int(line.split()[1])
                buf = defaultdict(list)
                field = None

            elif re.match(r"^\.[TWABXK]\b", line):
                field = line[1]

            elif field in ("T", "W"):
                buf[field].append(line)

    if cur_id is not None:
        items[cur_id] = " ".join(buf["T"] + buf["W"])

    return items


def load_qrels(path):
    """Load relevance judgments from CISI.REL."""
    rel = defaultdict(set)

    with open(path, encoding="latin-1") as f:
        for line in f:
            parts = line.split()

            if len(parts) >= 2:
                rel[int(parts[0])].add(int(parts[1]))

    return rel


# ============================================================
# 3. TEXT PREPROCESSING
# ============================================================

STOPWORDS = set("""
a about above after again against all am an and any are as at be
because been before being below between both but by can did do does
doing down during each few for from further had has have having he
her here hers herself him himself his how i if in into is it its
itself just me more most my myself no nor not now of off on once
only or other our ours ourselves out over own same she should so
some such than that the their theirs them themselves then there
these they this those through to too under until up very was we
were what when where which while who whom why will with would you
your yours yourself yourselves also may must shall upon within without
""".split())

try:
    from nltk.stem import PorterStemmer
    _stem = PorterStemmer().stem
except ImportError:
    def _stem(w):
        for suf in (
            "ational", "ization", "iveness", "fulness", "ousness",
            "ments", "ation", "ings", "ing", "ness", "ies",
            "ed", "es", "ly", "s"
        ):
            if w.endswith(suf) and len(w) - len(suf) >= 3:
                return w[:-len(suf)]
        return w


def preprocess(text):
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    return [
        _stem(t)
        for t in tokens
        if t not in STOPWORDS and len(t) > 1
    ]


# ============================================================
# 4. INVERTED INDEX
# ============================================================

class InvertedIndex:
    def __init__(self, docs):
        self.postings = defaultdict(dict)
        self.doc_len = {}

        for doc_id, text in docs.items():
            terms = preprocess(text)
            self.doc_len[doc_id] = len(terms)

            for term, tf in Counter(terms).items():
                self.postings[term][doc_id] = tf

        self.N = len(docs)
        self.avgdl = (
            sum(self.doc_len.values()) / self.N if self.N else 0
        )
        self.df = {
            term: len(postings)
            for term, postings in self.postings.items()
        }

    def idf(self, term):
        if term not in self.df:
            return 0.0
        return math.log(self.N / self.df[term])

    def bm25_idf(self, term):
        n = self.df.get(term, 0)
        return math.log(
            1 + (self.N - n + 0.5) / (n + 0.5)
        )


# ============================================================
# 5. TF-IDF VECTOR SPACE MODEL
# ============================================================

class TfIdfVSM:
    def __init__(self, index):
        self.ix = index
        self.norm = defaultdict(float)

        for term, postings in index.postings.items():
            idf = index.idf(term)

            for doc_id, tf in postings.items():
                weight = (1 + math.log10(tf)) * idf
                self.norm[doc_id] += weight ** 2

        self.norm = {
            doc_id: math.sqrt(value) or 1.0
            for doc_id, value in self.norm.items()
        }

    def search(self, query, k=None):
        q = Counter(preprocess(query))
        scores = defaultdict(float)
        qnorm = 0.0

        for term, qtf in q.items():
            if term not in self.ix.postings:
                continue

            idf = self.ix.idf(term)
            wq = (1 + math.log10(qtf)) * idf
            qnorm += wq ** 2

            for doc_id, tf in self.ix.postings[term].items():
                wd = (1 + math.log10(tf)) * idf
                scores[doc_id] += wq * wd

        qnorm = math.sqrt(qnorm) or 1.0

        ranked = sorted(
            [
                (
                    score / (qnorm * self.norm.get(doc_id, 1.0)),
                    doc_id
                )
                for doc_id, score in scores.items()
            ],
            reverse=True
        )

        results = [(doc_id, score) for score, doc_id in ranked]

        return results if k is None else results[:k]


# ============================================================
# 6. BM25 RETRIEVAL MODEL
# ============================================================

class BM25:
    def __init__(self, index, k1=1.5, b=0.75):
        self.ix = index
        self.k1 = k1
        self.b = b

    def search(self, query, k=None):
        scores = defaultdict(float)

        for term in set(preprocess(query)):
            if term not in self.ix.postings:
                continue

            idf = self.ix.bm25_idf(term)

            for doc_id, tf in self.ix.postings[term].items():
                dl = self.ix.doc_len[doc_id]

                if self.ix.avgdl == 0:
                    continue

                denom = tf + self.k1 * (
                    1 - self.b + self.b * dl / self.ix.avgdl
                )

                scores[doc_id] += (
                    idf * tf * (self.k1 + 1) / denom
                )

        ranked = sorted(
            scores.items(),
            key=lambda x: (-x[1], x[0])
        )

        return ranked if k is None else ranked[:k]


# ============================================================
# 7. EVALUATION METRICS
# ============================================================

def precision_at_k(ranked, rel, k):
    if k <= 0:
        return 0.0
    return sum(d in rel for d in ranked[:k]) / k


def recall_at_k(ranked, rel, k):
    if not rel or k <= 0:
        return 0.0
    return sum(d in rel for d in ranked[:k]) / len(rel)


def average_precision(ranked, rel):
    if not rel:
        return 0.0

    hits = 0
    total = 0.0

    for i, doc_id in enumerate(ranked, 1):
        if doc_id in rel:
            hits += 1
            total += hits / i

    return total / len(rel)


def ndcg_at_k(ranked, rel, k):
    dcg = sum(
        1 / math.log2(i + 1)
        for i, doc_id in enumerate(ranked[:k], 1)
        if doc_id in rel
    )

    ideal_hits = min(len(rel), k)

    idcg = sum(
        1 / math.log2(i + 1)
        for i in range(1, ideal_hits + 1)
    )

    return dcg / idcg if idcg else 0.0


def reciprocal_rank(ranked, rel):
    for i, doc_id in enumerate(ranked, 1):
        if doc_id in rel:
            return 1 / i
    return 0.0


def evaluate(model, queries, qrels, k=10):
    rows = []

    for qid, qtext in queries.items():
        rel = qrels.get(qid, set())

        # Skip queries without relevance judgments
        if not rel:
            continue

        ranked = [doc_id for doc_id, _ in model.search(qtext)]

        rows.append([
            precision_at_k(ranked, rel, k),
            recall_at_k(ranked, rel, k),
            average_precision(ranked, rel),
            ndcg_at_k(ranked, rel, k),
            reciprocal_rank(ranked, rel)
        ])

    if not rows:
        raise ValueError(
            "No queries with relevance judgments were evaluated. "
            "Check that the query IDs in CISI.QRY match CISI.REL."
        )

    means = np.mean(rows, axis=0)

    metric_names = [
        f"P@{k}", f"R@{k}", "MAP", f"nDCG@{k}", "MRR"
    ]

    return dict(zip(metric_names, map(float, means))), len(rows)


# ============================================================
# 8. MAIN PROGRAM
# ============================================================

def main():
    data_dir = Path(DATA_DIR)
    output_path = Path(OUTPUT_FILE)

    # Verify dataset files exist
    required_files = [
        data_dir / "CISI.ALL",
        data_dir / "CISI.QRY",
        data_dir / "CISI.REL"
    ]

    for file_path in required_files:
        if not file_path.is_file():
            raise FileNotFoundError(
                f"Missing file: {file_path}\n"
                "Upload CISI.ALL, CISI.QRY and CISI.REL "
                "to the configured DATA_DIR."
            )

    # Load dataset
    docs = parse_cisi_file(data_dir / "CISI.ALL")
    queries = parse_cisi_file(data_dir / "CISI.QRY")
    qrels = load_qrels(data_dir / "CISI.REL")

    print("=" * 65)
    print("CISI INFORMATION RETRIEVAL SYSTEM")
    print("=" * 65)

    print(f"Documents: {len(docs)}")
    print(f"Queries: {len(queries)}")
    print(f"Queries with judgments: {len(qrels)}")
    print(f"Judged query-document pairs: {sum(len(v) for v in qrels.values())}")

    if not docs or not queries or not qrels:
        raise ValueError(
            "One or more dataset files parsed as empty. "
            "Check the dataset format and file contents."
        )

    # Build inverted index
    index = InvertedIndex(docs)

    print(f"Vocabulary size: {len(index.postings)} terms")
    print(f"Average document length: {index.avgdl:.2f} tokens")

    # Create retrieval models
    models = {
        "TF-IDF (cosine)": TfIdfVSM(index),
        "BM25 (k1=1.5, b=0.75)": BM25(index)
    }

    # Evaluate both models
    results = {}

    print("\nEVALUATION RESULTS")
    print("-" * 75)

    metric_names = [
        f"P@{K}", f"R@{K}", "MAP", f"nDCG@{K}", "MRR"
    ]

    print(f"{'Model':<27}" + "".join(
        f"{name:>10}" for name in metric_names
    ))

    evaluated_count = 0

    for name, model in models.items():
        scores, n = evaluate(model, queries, qrels, K)
        evaluated_count = n
        results[name] = scores

        print(f"{name:<27}" + "".join(
            f"{value:>10.4f}" for value in scores.values()
        ))

    print(f"\nMetrics averaged over {evaluated_count} judged queries.")

    # BM25 parameter study
    print("\nBM25 PARAMETER STUDY (MAP)")
    print("-" * 40)

    sweep = {}

    for k1 in (0.9, 1.2, 1.5, 2.0):
        for b in (0.5, 0.75, 0.9):
            model = BM25(index, k1=k1, b=b)
            scores, _ = evaluate(model, queries, qrels, K)

            key = f"k1={k1}, b={b}"
            sweep[key] = scores["MAP"]

            print(f"{key:<20} MAP = {scores['MAP']:.4f}")

    results["bm25_sweep_MAP"] = sweep

    best_setting = max(sweep, key=sweep.get)
    print(f"\nBest tested BM25 setting by MAP: {best_setting}")
    print(f"Best MAP: {sweep[best_setting]:.4f}")

    # Optional custom search query
    if CUSTOM_QUERY:
        print(f"\nCUSTOM QUERY: {CUSTOM_QUERY}")

        for name, model in models.items():
            print(f"\nTop 5 results using {name}:")

            for rank, (doc_id, score) in enumerate(
                model.search(CUSTOM_QUERY, 5), 1
            ):
                text_preview = re.sub(
                    r"\s+", " ", docs[doc_id]
                )[:150]

                print(
                    f"{rank}. Document {doc_id} | "
                    f"Score: {score:.4f}\n"
                    f"   {text_preview}..."
                )

    # Save results as JSON
    output_path.parent.mkdir(parents=True, exist_ok=True)

    output_path.write_text(
        json.dumps(results, indent=2),
        encoding="utf-8"
    )

    print(f"\nResults saved to: {output_path}")
    print("\nProgram completed successfully!")


# ============================================================
# 9. RUN THE COMPLETE SYSTEM
# ============================================================

if __name__ == "__main__":
    main()

CISI INFORMATION RETRIEVAL SYSTEM
Documents: 1460
Queries: 112
Queries with judgments: 76
Judged query-document pairs: 3114
Vocabulary size: 6027 terms
Average document length: 71.10 tokens

EVALUATION RESULTS
---------------------------------------------------------------------------
Model                            P@10      R@10       MAP   nDCG@10       MRR
TF-IDF (cosine)                0.3158    0.1300    0.2028    0.3495    0.5893
BM25 (k1=1.5, b=0.75)          0.3079    0.1101    0.1815    0.3381    0.5754

Metrics averaged over 76 judged queries.

BM25 PARAMETER STUDY (MAP)
----------------------------------------
k1=0.9, b=0.5        MAP = 0.1680
k1=0.9, b=0.75       MAP = 0.1741
k1=0.9, b=0.9        MAP = 0.1756
k1=1.2, b=0.5        MAP = 0.1723
k1=1.2, b=0.75       MAP = 0.1790
k1=1.2, b=0.9        MAP = 0.1794
k1=1.5, b=0.5        MAP = 0.1748
k1=1.5, b=0.75       MAP = 0.1815
k1=1.5, b=0.9        MAP = 0.1817
k1=2.0, b=0.5        MAP = 0.1798
k1=2.0, b=0.75       MAP = 0.